# 📖 Tutorial 04: AlphaZero Training Pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lessen-xu/Hybrid-Chess/blob/main/notebooks/04_alphazero_training.ipynb)

---

## Who is this for?

You've learned search (Tutorial 02) and MCTS (Tutorial 03). Now we put it all together: a neural network that **learns to play from scratch** through self-play. This is the full AlphaZero pipeline.

**What you'll learn:**
1. The AlphaZero training loop: **self-play → train → evaluate**
2. The **neural network architecture**: dual-head ResNet (policy + value)
3. How **state encoding** works (board → tensor)
4. How **action encoding** maps moves to network outputs
5. The **self-play** process: MCTS + temperature + resign + draw adjudication
6. The **training loss**: policy cross-entropy + value MSE
7. **Gating**: how we decide whether a new model is actually better

**Prerequisites:** Tutorials 01-03, basic PyTorch (tensors, forward pass).

---

## 0. Setup

In [ ]:
# Uncomment for Google Colab:
# !pip install -q git+https://github.com/lessen-xu/Hybrid-Chess.git

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from hybrid.core.env import HybridChessEnv, GameState
from hybrid.core.types import Side, Move, PieceKind
from hybrid.core.render import render_board
from hybrid.core.config import BOARD_H, BOARD_W

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Ready — device: {device}")

---

## 1. The AlphaZero Loop

### 1.1 Theory: How AlphaZero learns

AlphaZero starts with a randomly initialized neural network that knows **nothing** about the game. Through an iterative process, it bootstraps its own knowledge:

```
  ┌──────────────────────────────────────────────────────────┐
  │                  AlphaZero Training Loop                 │
  │                                                          │
  │   ① Self-Play         ② Train           ③ Evaluate      │
  │   ──────────         ─────────         ──────────        │
  │   Current model      Fit network       New model vs     │
  │   plays itself       to self-play      current model    │
  │   N games via        data: minimize    in M head-to-    │
  │   MCTS               policy CE +       head matches     │
  │                      value MSE                          │
  │        │                  │                 │            │
  │        ▼                  ▼                 ▼            │
  │   (s, π, z)          Updated            Accept or       │
  │   training           weights            reject new      │
  │   examples                              model           │
  │                                                          │
  │   ◄─────────── Repeat for K iterations ───────────►     │
  └──────────────────────────────────────────────────────────┘
```

Each training example from self-play is a tuple **(s, π, z)**:
- **s** = the encoded board state (input to the network)
- **π** = MCTS visit distribution (the "improved" policy — the search-refined target)
- **z** = the game outcome (+1 win, −1 loss, 0 draw) from this position's perspective

### 1.2 Why self-play works

The magic of AlphaZero is that it doesn't need human games, openings databases, or expert knowledge. It generates its own training data:

1. The network plays MCTS-guided games against **itself**
2. MCTS search produces a **better policy** than the raw network (because search looks ahead)
3. We train the network to match this better policy → the network improves
4. A better network → better MCTS → even better policy → repeat

This is called **policy improvement through search** — a beautiful idea from reinforcement learning.

---

## 2. State Encoding: Board → Tensor

### 2.1 Theory: How to represent a board for a neural network

Neural networks work with tensors (multi-dimensional arrays of numbers). We need to convert a board position into a fixed-size tensor.

We use **binary feature planes** — a 3D tensor of shape `(C, H, W)` where:
- `H × W` = 10 × 9 (board dimensions)
- `C` = 14 channels (13 piece types + 1 side-to-move indicator)

Each channel is a binary grid: 1 where that piece type exists, 0 elsewhere.

```
Channel 0: King positions        Channel 6: General positions
  . . . . . . . . .              . . . . . . . . .
  . . . . . . . . .              . . . . . . . . .
  . . . . . . . . .              . . . . . . . . .
  . . . . . . . . .              . . . . 1 . . . .  ← General at (4,9)
  . . . . 1 . . . .  ← King     . . . . . . . . .
  ...                            ...

Channel 13: Side to move
  1 1 1 1 1 1 1 1 1              (all 1s if Chess to move)
  1 1 1 1 1 1 1 1 1              (all 0s if Xiangqi to move)
  ...
```

### 2.2 Why this encoding?

- **Translation equivariance**: Convolutional layers naturally learn spatial patterns
- **Fixed size**: Always (14, 10, 9) regardless of how many pieces are on the board
- **Complete information**: Encodes everything about the position

In [ ]:
from hybrid.rl.az_encoding import encode_state, NUM_STATE_CHANNELS

# Encode the opening position
env = HybridChessEnv()
state = env.reset()

tensor = encode_state(state)
print(f"Encoded state shape: {tensor.shape}")
print(f"  → {NUM_STATE_CHANNELS} channels × {BOARD_H} rows × {BOARD_W} cols")
print(f"  → dtype: {tensor.dtype}")
print(f"  → Total elements: {tensor.numel()}")

# Show which channels have pieces
print(f"\nPieces per channel:")
channel_names = [
    "King", "Queen", "Rook", "Bishop", "Knight", "Pawn",
    "General", "Advisor", "Elephant", "Horse", "Chariot", "Cannon", "Soldier",
    "Side-to-move",
]
for i, name in enumerate(channel_names):
    count = tensor[i].sum().item()
    if count > 0:
        print(f"  Ch {i:>2} ({name:>12}): {int(count)} squares")

---

## 3. Neural Network: PolicyValueNet

### 3.1 Theory: The dual-head architecture

AlphaZero uses a single network with **two output heads**:

```
  Input: (B, 14, 10, 9)   ← encoded board state
         ↓
  Conv 3×3 (14→64) + BatchNorm + ReLU
         ↓
  ResBlock × 3  [Conv→BN→ReLU→Conv→BN + Skip→ReLU]
         ↓                          ↓
  ┌─── Policy Head ───┐   ┌─── Value Head ────┐
  │ Conv 1×1 (64→92)  │   │ Conv 1×1 (64→1)   │
  │ → (B, 92, 10, 9)  │   │ → BN → flatten    │
  │ = logits per move  │   │ → FC(90→64)→ReLU  │
  │                    │   │ → FC(64→1)→tanh   │
  └────────────────────┘   └────────────────────┘
     π(a|s) ∈ ℝ^8280          v(s) ∈ [-1, +1]
```

**Why two heads?**
- The **policy head** outputs a probability distribution over all possible moves. This guides MCTS search (which moves to explore first).
- The **value head** outputs a single scalar: how good is this position? (+1 = winning, -1 = losing). This replaces the random rollout in MCTS.

**Why share the body?** Both tasks (predicting good moves and evaluating positions) require similar spatial understanding of the board. Sharing the convolutional layers lets the network develop shared representations.

### 3.2 Residual blocks

The **residual connection** (skip connection) is a key architectural innovation:

```
  input ──┬──→ Conv→BN→ReLU→Conv→BN ──┬──→ ReLU → output
          │                             │
          └─────── skip connection ─────┘
```

The skip connection lets gradients flow directly backward, making it possible to train deeper networks. The network can learn to "pass through" the residual block if it's not helpful (output = input + 0), which makes optimization easier.

In [ ]:
from hybrid.rl.az_network import PolicyValueNet

# Create a network (default: 64 channels, 3 res blocks)
net = PolicyValueNet(
    in_channels=NUM_STATE_CHANNELS,  # 14
    num_channels=64,
    num_res_blocks=3,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in net.parameters())
trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)

print(f"PolicyValueNet architecture:")
print(f"  Input:    (B, {NUM_STATE_CHANNELS}, {BOARD_H}, {BOARD_W})")
print(f"  Channels: 64")
print(f"  Res blocks: 3")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

# Forward pass
x = tensor.unsqueeze(0).float().to(device)  # add batch dim
with torch.no_grad():
    policy_logits, value = net(x)

print(f"\nForward pass:")
print(f"  Policy logits shape: {policy_logits.shape}  (92 planes × 10 × 9)")
print(f"  Value shape:         {value.shape}")
print(f"  Value (untrained):   {value.item():.4f}  (random, close to 0)")

# Policy distribution
policy_flat = policy_logits.view(-1)
policy_probs = F.softmax(policy_flat, dim=0)
print(f"  Policy: {policy_probs.shape[0]} action probabilities")
print(f"  Max prob: {policy_probs.max().item():.4f} (should be ~uniform for untrained net)")

---

## 4. Action Encoding: Moves → Network Outputs

### 4.1 Theory: Mapping moves to a fixed-size output

The policy head outputs a tensor of shape `(92, 10, 9)`. Each of the 8,280 elements corresponds to one possible move:

```
92 planes = 72 sliding + 8 knight + 12 promotion

Sliding (72 planes):
  8 directions × 9 max distances = 72
  Directions: N, NE, E, SE, S, SW, W, NW
  Distance: 1-9 squares
  → Covers all Rook, Bishop, Queen, Chariot, Cannon moves

Knight/Horse (8 planes):
  8 L-shaped deltas
  → Covers Knight and Horse moves

Promotion (12 planes):
  3 forward directions × 4 piece types
  → For Pawn promotion at the top row
```

The position `(plane, from_y, from_x)` encodes: "move the piece at `(from_x, from_y)` in the direction/distance specified by `plane`."

Most of these 8,280 actions are illegal at any given position. We **mask** them during MCTS to only consider legal moves.

In [ ]:
from hybrid.rl.az_encoding import move_to_plane, TOTAL_POLICY_PLANES

print(f"Total policy planes: {TOTAL_POLICY_PLANES}")
print(f"Action space size: {TOTAL_POLICY_PLANES} × {BOARD_H} × {BOARD_W} = "
      f"{TOTAL_POLICY_PLANES * BOARD_H * BOARD_W}")

# Let's decode a specific move
env = HybridChessEnv()
state = env.reset()
legal = env.legal_moves()

print(f"\nEncoding example moves:")
for mv in legal[:5]:
    piece = state.board.get(mv.fx, mv.fy)
    plane_idx, ey, ex = move_to_plane(mv)
    flat_idx = plane_idx * (BOARD_H * BOARD_W) + ey * BOARD_W + ex
    print(f"  {piece.kind.name:>8} ({mv.fx},{mv.fy})→({mv.tx},{mv.ty}) "
          f"→ plane={plane_idx}, pos=({ex},{ey}), flat={flat_idx}")

print(f"\n💡 Only {len(legal)} of {TOTAL_POLICY_PLANES * BOARD_H * BOARD_W} "
      f"actions are legal ({len(legal)/8280*100:.1f}%)")

---

## 5. Self-Play: Generating Training Data

### 5.1 Theory: The self-play process

Self-play generates training data by having the current network play against itself:

```
  For each game:
    1. Reset the environment
    2. At each move:
       a. Run MCTS with the current network (N simulations)
       b. Get visit distribution π from MCTS root
       c. Choose move based on π (with temperature)
       d. Record training example: (state, π, side_to_move)
    3. When game ends, assign z to all examples:
       z = +1 for winning side, -1 for losing side, 0 for draw
```

### 5.2 Temperature schedule

**Temperature** controls exploration during move selection from the MCTS visit distribution:

| Temperature | Effect | When |
|------------|--------|------|
| T = 1.0 | Proportional to visits (exploratory) | First 30 moves (opening) |
| T → 0 | Always pick the most visited move (greedy) | After move 30 (tactical play) |

High temperature in the opening creates **diverse training data**. Low temperature later ensures the network plays its best moves (avoiding blunders in critical positions).

### 5.3 Resign and draw adjudication

To speed up self-play, two mechanisms cut games short:

- **Resign**: If the root value is below −0.95 for 3 consecutive moves (after ply 40), the losing side resigns
- **Draw adjudication**: If |root value| < 0.08 for 15 consecutive moves (after ply 60), the game is declared a draw

These save compute by avoiding long, foreordained endgames.

In [ ]:
from hybrid.rl.az_selfplay import SelfPlayConfig

cfg = SelfPlayConfig()
print("Default SelfPlayConfig:")
print(f"  temperature:         {cfg.temperature}")
print(f"  temp_cutoff_ply:     {cfg.temp_cutoff_ply}  (T=1.0 for first {cfg.temp_cutoff_ply} plies)")
print(f"  simulations:         {cfg.simulations}  (MCTS sims per move)")
print(f"  max_ply:             {cfg.max_ply}")
print(f"  resign_enabled:      {cfg.resign_enabled}")
print(f"  resign_threshold:    {cfg.resign_threshold}")
print(f"  resign_patience:     {cfg.resign_patience}")
print(f"  draw_adjudicate:     {cfg.draw_adjudicate_enabled}")
print(f"  draw_adj_patience:   {cfg.draw_adjudicate_patience}")
print(f"  reward_shaper:       {cfg.reward_shaper}  (None = pure AlphaZero)")

---

## 6. Training: Learning from Self-Play Data

### 6.1 Theory: The loss function

The network is trained to simultaneously predict:
1. The MCTS policy (which moves to explore)
2. The game outcome (who wins from this position)

$$\mathcal{L} = \underbrace{-\pi^\top \log p}_{\text{policy loss (cross-entropy)}} + \underbrace{(z - v)^2}_{\text{value loss (MSE)}}$$

Where:
- $\pi$ = MCTS visit distribution (target policy from self-play)
- $p$ = network's policy output (softmax of policy logits)
- $z$ = actual game outcome (+1/−1/0)
- $v$ = network's value prediction

### 6.2 Why cross-entropy for policy?

Cross-entropy loss encourages the network to assign high probability to moves that MCTS visited frequently. It works with the **soft** MCTS distribution, not just the best move:

```
MCTS visits:  [Move A: 50, Move B: 30, Move C: 15, Move D: 5]
Target π:     [0.50,       0.30,       0.15,       0.05      ]

Network should learn to assign:  P(A) ≈ 0.50, P(B) ≈ 0.30, etc.
```

This is better than a hard target ("Move A is best") because the distribution captures the **relative quality** of all explored moves.

### 6.3 Implementation detail: Sparse π

Our implementation stores π in **sparse format** (only the non-zero entries) to save memory:

```python
pi_indices = [42, 156, 203, 891]   # which actions have non-zero probability
pi_probs   = [0.50, 0.30, 0.15, 0.05]  # their probabilities
```

During training, we gather the corresponding logits, apply log_softmax, and compute cross-entropy only over these entries.

In [ ]:
# Demonstrate one training step (with synthetic data)

net = PolicyValueNet(in_channels=14, num_channels=64, num_res_blocks=3).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)

# Create a fake mini-batch of 4 examples
batch_states = torch.randn(4, 14, 10, 9, device=device)  # random encoded states
batch_z = torch.tensor([1.0, -1.0, 0.0, 1.0], device=device)  # game outcomes

# Fake sparse π (4 legal moves each)
fake_pi_indices = [np.array([10, 20, 30, 40], dtype=np.uint16)] * 4
fake_pi_probs = [np.array([0.4, 0.3, 0.2, 0.1], dtype=np.float32)] * 4

# Forward pass
net.train()
policy_logits, value = net(batch_states)

# Policy loss (simplified)
B = len(batch_z)
policy_flat = policy_logits.view(B, -1)  # (4, 8280)

# Gather logits at legal indices
max_L = 4
padded_idx = torch.zeros((B, max_L), dtype=torch.long, device=device)
padded_probs = torch.zeros((B, max_L), dtype=torch.float32, device=device)
for i in range(B):
    padded_idx[i] = torch.as_tensor(fake_pi_indices[i].astype(np.int64))
    padded_probs[i] = torch.as_tensor(fake_pi_probs[i])

gathered = policy_flat.gather(1, padded_idx)
log_probs = F.log_softmax(gathered, dim=1)
policy_loss = -(padded_probs * log_probs).sum(dim=1).mean()

# Value loss
value_loss = F.mse_loss(value.squeeze(-1), batch_z)

# Total loss
loss = policy_loss + value_loss

# Backward + update
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f"Training step result:")
print(f"  Policy loss: {policy_loss.item():.4f}")
print(f"  Value loss:  {value_loss.item():.4f}")
print(f"  Total loss:  {loss.item():.4f}")
print(f"\n💡 Policy loss starts high (network output is random, not matching π)")
print(f"   After many iterations of self-play + train, both losses decrease.")

---

## 7. Gating: Accepting or Rejecting New Models

### 7.1 Theory: Why gating matters

After training, we have a new model. But is it actually **better** than the current best? Random optimization noise could make a model temporarily worse.

**Gating** is a quality control step:
1. Play M matches between the **new model** and the **current best model**
2. Compute a statistical test (Wilson confidence interval) on the match results
3. Accept the new model **only if it wins significantly more than 50%**

### 7.2 Wilson confidence interval

The Wilson score interval gives a confidence bound on the true win rate given observed results:

$$\hat{p}_{\text{lower}} = \frac{\hat{p} + \frac{z^2}{2n} - z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$$

If the lower bound of the confidence interval is above 0.5 (with draws scored as 0.5), we accept the new model.

### 7.3 Why not just use raw win rate?

With small sample sizes (e.g., 20 games), random variance is high. A model might win 12/20 (60%) by chance even if it's not truly better. The Wilson interval accounts for this uncertainty — it requires **statistically significant** improvement.

In [ ]:
# Wilson confidence interval demo
import math

def wilson_lower(wins, total, z=1.96):  # z=1.96 for 95% confidence
    """Lower bound of Wilson score interval."""
    if total == 0:
        return 0.0
    p_hat = wins / total
    denom = 1 + z**2 / total
    center = p_hat + z**2 / (2 * total)
    margin = z * math.sqrt(p_hat * (1 - p_hat) / total + z**2 / (4 * total**2))
    return (center - margin) / denom

print("Gating examples (95% confidence):")
print(f"{'Wins/Total':<15} {'Raw WR':>8} {'Wilson Lower':>13} {'Accept?':>8}")
print("─" * 50)

cases = [
    (6, 10), (7, 10), (8, 10),       # small sample
    (12, 20), (13, 20), (15, 20),     # medium sample
    (55, 100), (60, 100), (65, 100),  # large sample
]
for w, t in cases:
    lower = wilson_lower(w, t)
    accept = "✅ Yes" if lower > 0.5 else "❌ No"
    print(f"{w}/{t:<13} {w/t:>7.1%} {lower:>12.3f}  {accept:>8}")

print(f"\n💡 With only 10 games, you need 8+ wins to be confident.")
print(f"   With 100 games, 60+ wins is enough.")

---

## 8. Running a Mini Training

### 8.1 Train from the command line

The easiest way to run a full training loop:

```bash
# Quick test (CPU, ~5 min)
python -m hybrid train --iterations 5 --games 20 --simulations 50

# Full training (GPU recommended)
python -m hybrid train \
    --iterations 100 --games 200 --simulations 200 \
    --device cuda --workers 8 --use-cpp \
    --output runs/experiment_1
```

### 8.2 What to watch

During training, key metrics to monitor:

| Metric | What it means | Healthy trend |
|--------|--------------|---------------|
| `policy_loss` | Network's policy vs MCTS policy | Decreasing |
| `value_loss` | Network value vs game outcome | Decreasing |
| `gate_accepted` | New model beat old model? | Yes (most of the time) |
| `vs_random_wr` | Win rate against random | Increasing → ~95%+ |
| `vs_ab_d1_wr` | Win rate against AlphaBeta d=1 | Increasing |
| `avg_game_length` | Self-play avg ply count | Decreasing (winning faster) |

---

## 🎓 Summary

| Concept | What we learned |
|---------|----------------|
| **AlphaZero loop** | Self-play → Train → Evaluate, repeat |
| **Training examples** | (state, π, z) — position, MCTS policy, outcome |
| **State encoding** | 14 binary planes: 13 piece types + side-to-move |
| **Action encoding** | 92 planes × 10×9 = 8,280 possible actions |
| **PolicyValueNet** | Dual-head ResNet: shared body → policy head + value head |
| **Policy loss** | Cross-entropy with MCTS visit distribution |
| **Value loss** | MSE between predicted and actual game outcome |
| **Temperature** | T=1.0 for exploration (opening), T→0 for exploitation (later) |
| **Gating** | Wilson CI: accept new model only with statistical significance |

## ⏭️ What's Next?

- **Tutorial 05: Experiments** — Customize everything: variants, reward shaping, architectures, curriculum